# 01 · Band-Split Architecture

**Does giving frequency bands their own encoder capacity help — at *matched*
parameters?** This notebook is the visual companion to
[`../THEORY.md`](../THEORY.md): it shows the two band layouts, walks through the
3-tower architecture (per-band encoders → frequency-concatenated skips → the
baseline decoder), renders the **exact parameter match** (the credibility crux),
and draws the capacity-vs-compute distinction. It sets up the experiments in
[`02_bandsplit_experiments.ipynb`](02_bandsplit_experiments.ipynb).

Nothing here needs a GPU. Most cells run on CPU from the model definitions alone;
the two that need decoded MUSDB shards (the real spectrogram overlay, the
vocal-energy-by-band table) are marked **⚠️ RUN THIS LATER**. The notebook ships
**un-executed** so the committed file is a clean scaffold.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §3.1 (the three arms, the
  band-split architecture, the width search), §6.4 (efficiency), §9 (code
  contracts); [`../THEORY.md`](../THEORY.md) §1 (weight-sharing scope), §2 (mel
  edges), §3 (boundary analysis), §4 (parameter & compute accounting — the crux).
- **Data prep is *not* repeated here.** Acquisition, licensing, the 86/14/50
  split, and the STFT/chunking front end live in Direction 01's notebooks
  ([`../../01-loss-function-study/notebooks/01_data_and_eda.ipynb`](../../01-loss-function-study/notebooks/01_data_and_eda.ipynb),
  `02_pipeline_and_model.ipynb`). This direction reuses them verbatim.
- **Code, not prose, is authoritative:** the model is
  `singnet/models/bandsplit_unet.py`; the width search is
  `scripts/match_params.py`; every number below is asserted in `tests/` (gate G0).

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · The idea in one picture — two ways to cut the spectrum

The 2048 network bins are split into **3 contiguous bands**, each with its own
encoder tower. Two layouts (only the edges differ):

| Layout | Interior edges (bins) | Interior edges (Hz) | Rationale |
|---|---|---|---|
| **mel** | 142, 597 | ≈ 1.53 kHz, 6.43 kHz | narrow low/mid bands where vocal energy concentrates |
| **uniform** | 683, 1365 | ≈ 7.35 kHz, 14.70 kHz | equal **bin** widths — the control isolating *splitting* from *mel spacing* |

The mel layout spends two of three bands below 6.4 kHz (matching sung
fundamentals + formants, ~100 Hz–4 kHz); the uniform control spends two thirds of
its bins above 7.35 kHz. That difference is exactly what the $E_2$ contrast tests
(notebook 02).

In [ ]:
# CPU-runnable later: derive both band layouts from code and show them.
from singnet.models import mel_edges, uniform_edges
from singnet.models.bandsplit_unet import bin_to_hz

for name, edges in [("mel", mel_edges(3)), ("uniform", uniform_edges(3))]:
    hz = [round(bin_to_hz(b)) for b in edges]
    widths = [hi - lo for lo, hi in zip(edges, edges[1:])]
    print(f"{name:8} edges(bins)={edges}  edges(Hz)={hz}  band widths(bins)={widths}")
# Both partitions are exact / disjoint / exhaustive over the 2048 bins (asserted in
# tests/test_bandsplit_model.py). Figure (RUN LATER, CPU): overlay these edges as
# horizontal lines on a log-mel spectrogram of a validation vocal to see how each
# layout carves the vocal energy.

In [ ]:
# ⚠️ RUN THIS LATER (CPU, needs one decoded shard) — the layout overlay figure.
# import os, torch, matplotlib.pyplot as plt
# from singnet.data import WavShardStore
# from singnet.audio import STFT
# from singnet.models import mel_edges, uniform_edges
# store = WavShardStore(os.environ["SHARD_ROOT"])
# voc = torch.from_numpy(store.load_sources(store.track_names()[0])["vocals"]).float()
# mag = STFT().transform(voc[: 261120]).abs()[:2048]        # (2048, T)
# plt.imshow(torch.log1p(mag).numpy(), origin="lower", aspect="auto")
# for b in mel_edges(3)[1:-1]:     plt.axhline(b, color="C1", lw=2, label="mel")
# for b in uniform_edges(3)[1:-1]: plt.axhline(b, color="C2", ls="--", lw=2, label="uniform")
# plt.title("Band layouts over a vocal spectrogram"); plt.legend()
print("Layout-overlay figure renders from a shard (RUN LATER).")

## 2 · Where vocal energy lives (and the AAC caveat)

The mel layout is motivated by *where the energy is*. The **vocal-energy-by-band**
table (below, RUN LATER) quantifies, from the prep-time energy index, how much
vocal energy falls in each band of each layout — the motivating figure for the
mechanism analysis.

**Design-honesty caveat (stated up front):** MUSDB18's sources are AAC-encoded and
band-limited to ≈ 16 kHz, so the mel **top** band [6.43, 22.05] kHz contains a
large dead sub-region [16, 22] kHz. Capacity is "wasted" there *by design* — a
property of mel spacing on this data, part of the treatment (THEORY §2.2). The
banded metrics NaN-guard that region (notebook 02 §4).

In [ ]:
# ⚠️ RUN THIS LATER (CPU, ~minutes) — vocal-energy-by-band from the prep index.
# The energy index is written by scripts/prepare_data.py; here we bucket per-band
# reference vocal energy for both layouts so the mechanism figure has a prior.
# import json, os
# from singnet.eval.banded import analysis_grid_hz
# idx = json.load(open(os.path.join(os.environ["SHARD_ROOT"], "index.json")))
# ... aggregate per-band vocal energy over the 14 validation tracks ...
print("Vocal-energy-by-band table computes from the prep index (RUN LATER).")

## 3 · Architecture walkthrough

Each band gets its own **5-level** encoder tower (the *same* `Conv(5×5,s2)-BN-LReLU`
blocks as the baseline, reused verbatim from `singnet/models/unet.py`), base width
`c` common to all towers, channels `c → 2c → 4c → 8c → b5`. Then:

1. **Pad/crop** — a tower pads its band up to the next multiple of 32 (so five
   stride-2 layers halve exactly) and the padding is cropped back off the final
   mask. Edges therefore need no divisibility.
2. **No cross-band mixing before the bottleneck** — the towers are independent.
3. **Frequency-axis concatenation** — at each level the towers' outputs are
   concatenated along frequency (channel counts already match by the shared `c`),
   forming full-spectrum skips.
4. **Baseline-identical bottleneck + decoder** at width `c` (dec1 consumes the
   `b5` bottleneck), and a baseline-identical mask head + Nyquist handling — so
   `BandSplitUNet` is a **drop-in** for `SingNetC1` on the `(B,2048,256)` interface.

In [ ]:
# CPU-runnable later: instantiate both variants and inspect the internal shapes.
import torch
from singnet.models import BandSplitUNet

for name, model in [("mel", BandSplitUNet.from_mel_bands()),
                    ("uniform", BandSplitUNet.from_uniform_bands())]:
    print(f"\n{name}: edges={model.edges_bins}  base_width c={model.base_width}  "
          f"bottleneck b5={model.bottleneck_width}")
    print(f"   per-band widths {model.band_widths} -> padded {model.padded_widths} "
          f"(each a multiple of 32); padded total P={model.padded_total}")
    x = torch.rand(1, 2048, 256)                       # a (B, 2048, 256) mixture magnitude
    print(f"   forward: {tuple(x.shape)} -> {tuple(model(x).shape)}  (drop-in for SingNetC1)")

## 4 · The parameter match — the credibility crux

A SI-SDR gap between arms of *different* size would be a capacity artifact, not a
band-split effect. `scripts/match_params.py` derives, in closed form and then
cross-checks against the built `nn.Module`, the shared width that lands within
**±2 %** of the 9,835,745-param baseline. Result (THEORY §4):

- largest base width whose *pure* variant is under budget: **c = 23** (pure
  9,584,170, −2.56 % — outside 2 %),
- a single bottleneck bump **Δ = +14 → b5 = 382** closes the gap,
- matched variant: **9,841,896** params, **+0.0625 %** (both mel and uniform have
  this *identical* count — conv params are spatial-size-independent, so the band
  edges change compute, not parameters).

In [ ]:
# CPU-runnable later: render the committed per-module match table and SHOW the ±2%.
import sys; sys.path.insert(0, "scripts")
import match_params as mp
r = mp.match_width()
actual = mp.verify_against_model(r)     # asserts closed form == built model, else raises
print(f"c={r.base_width}  b5={r.bottleneck_width}  delta=+{r.delta}  "
      f"params={r.params:,}  rel_err={r.rel_error:+.4%}  within_2pct={r.within_tol}")
assert r.within_tol and abs(r.rel_error) <= 0.02      # the +-2% assertion, shown
print(mp.render_markdown(r, actual))                  # same table as results/param_match_table.md

## 5 · Capacity ≠ compute (the FLOPs bonus)

Conv **parameters** are independent of spatial size, but **MACs** are not. The
baseline encoder runs on all 2048 bins; each variant tower runs on ≈ 1/3 of them,
and the 3× tower count *cancels* the 1/3 spatial extent — so at **matched
parameters** the variants cost ≈ **0.54×** the baseline's encoder MACs (≈ 1.85×
cheaper). This is an **incidental efficiency bonus**, reported honestly as a
descriptive secondary — never the claim (the hypothesis is decided at matched
*parameters*; THEORY §4.5). Note mel and uniform have the *same* MACs (both depend
only on the padded total P = 2112), so the two variants are matched in compute too.

In [ ]:
# CPU-runnable later: the analytic MAC accounting from THEORY §4.5 (a conv layer
# costs ~ S_out * 25 * c_in * c_out MACs). Measured MACs on a real 6-s chunk (a
# forward hook) are a RUN-LATER descriptive secondary (MASTER_PLAN §6.4).
F, T = 2048, 256
def enc_macs(widths, freq_extent):
    macs, f, t = 0, freq_extent, T
    for l in range(5):
        f //= 2; t //= 2
        macs += f * t * 25 * widths[l] * widths[l + 1]
    return macs
base = enc_macs([1, 32, 64, 128, 256, 512], F)
w = [1, 23, 46, 92, 184, 382]
var = sum(enc_macs(w, p) for p in [160, 480, 1472])    # mel padded widths; uniform gives the same
print(f"baseline encoder MACs {base:,}")
print(f"variant  encoder MACs {var:,}   ratio {var/base:.3f}  (~2x cheaper at matched params)")

## 6 · Routing round-trip demo

The pad→crop routing must recover exactly the 2048 real bins in ascending order
(the padding is dropped, the bands reassembled). This is unit-tested
(`tests/test_bandsplit_model.py::test_routing_roundtrip_*`); the cell below is the
CPU-runnable demo.

In [ ]:
# CPU-runnable later: pad then crop-back is exact (identity on the real bins).
import torch
from singnet.models import BandSplitUNet
model = BandSplitUNet.from_mel_bands()
x = torch.randn(2, 1, 2048, 256)                       # a (B, C, F, T) feature map
recovered = model.crop_back(torch.cat(model.split_and_pad(x), dim=-2))
print("routing round-trip exact:", torch.equal(recovered, x))
assert torch.equal(recovered, x)

## 7 · The boundary-artifact question — what a tower cannot see

Each tower has the baseline receptive field (5, 13, 29, 61, 125 bins across the
five levels) **but is confined to its band**: it cannot see across a band edge. A
spectral structure straddling a boundary (e.g. a vocal harmonic near the mel edge
at bin 142 ≈ 1.53 kHz) is split between two towers that cannot share it in the
encoder, whereas the baseline sees it freely. Cross-band information returns only
in the shared decoder. **This severing is the mechanism for a pre-registered
"refuted (negative)" outcome** (THEORY §3): band isolation can cost more than the
dedicated capacity buys. Notebook 02's per-band figure localizes any damage at
band-boundary-adjacent bands vs. band interiors.

In [ ]:
# CPU-runnable later: the per-tower encoder receptive field, and the band
# boundaries the towers cannot cross.
# receptive field of 5x5 stride-2 layers: r_l = r_{l-1} + (k-1)*prod(strides<l)
r, rf = 1, []
for l in range(5):
    r = r + (5 - 1) * (2 ** l); rf.append(r)
print("per-tower receptive field by level (bins):", rf)
print("-> a tower sees <=125 bins and never across its band edge until the shared decoder.")

## 8 · Takeaways

- **Splitting = a change of weight-sharing scope** — per-band conv capacity instead
  of one full-spectrum filter bank; the sequence model of the SOTA family is
  *dropped* so the partition is isolated.
- **The match is exact, not hand-waved** — c = 23, b5 = 382, 9,841,896 params
  (+0.06 %), identical for mel and uniform, asserted against the built model.
- **Capacity ≠ compute** — matched params, yet ≈ 0.54× encoder MACs (a bonus,
  never the claim).
- **The risk is boundaries** — towers cannot see across band edges; the per-band
  figure is the pre-registered diagnostic.

Next: [`02_bandsplit_experiments.ipynb`](02_bandsplit_experiments.ipynb) runs the
6 split jobs (mel ×3, uniform ×3), reuses the shared baseline cell, reads the
ordered chain mel > uniform > baseline against the σ_seed band, and draws the
per-band mechanism figure.